# Kalshi Available Markets Exploration

In [8]:
import os
print("Your file is saved exactly at:")
os.getcwd()

target_dir = "C:/Python/CSUREMM/kalshi_data"
os.chdir(target_dir)
os.getcwd()

Your file is saved exactly at:


'C:\\Python\\CSUREMM\\kalshi_data'

In [9]:
import requests
import pandas as pd
import time
import os

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
OUTPUT_FILE = "kalshi_historical_markets.csv"
CHUNK_SIZE = 50   # Pages per disk flush
LIMIT = 100       # Kalshi's real effective max per page

In [10]:
def get_markets(cursor=None, limit=LIMIT):
    params = {"limit": limit}
    if cursor:
        params["cursor"] = cursor

    r = requests.get(
        f"{BASE_URL}/markets",
        params=params,
        timeout=30
    )
    r.raise_for_status()
    return r.json()

In [16]:
r = requests.get(f"{BASE_URL}/historical/cutoff")
print(r.json())
# → {"market_settled_ts": "2025-XX-XXTXX:XX:XXZ", ...}

{'market_settled_ts': '2026-04-14T00:00:00Z', 'orders_updated_ts': '2026-04-14T00:00:00Z', 'trades_created_ts': '2026-04-14T00:00:00Z'}


In [ ]:
def get_historical_markets(cursor=None, limit=LIMIT):  # GET /historical/markets
    params = {"limit": limit}
    if cursor:
        params["cursor"] = cursor
    r = requests.get(f"{BASE_URL}/historical/markets", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [11]:
def get_all_markets():
    cursor = None
    page = 1
    current_chunk = []
    first_write = True  # Track whether to write CSV header

    # Clear stale output file before starting fresh
    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)
        print(f"Removed old {OUTPUT_FILE}")

    while True:
        print(f"Downloading page {page}...")

        # --- Retry loop: keeps going on transient errors ---
        while True:
            try:
                data = get_markets(cursor=cursor, limit=LIMIT)
                break  # Success — exit retry loop
            except requests.exceptions.RequestException as e:
                print(f"Error on page {page}: {e}. Retrying in 5s...")
                time.sleep(5)

        markets = data.get("markets", [])

        if not markets:
            print("No more markets returned. Done!")
            break

        current_chunk.extend(markets)

        next_cursor = data.get("cursor")

        print(f"  page markets: {len(markets)} | chunk: {len(current_chunk)} | next_cursor: {bool(next_cursor)}")

        # --- Periodic flush to disk every CHUNK_SIZE pages ---
        if page % CHUNK_SIZE == 0:
            df_chunk = pd.DataFrame(current_chunk)
            df_chunk.to_csv(OUTPUT_FILE, mode='a', header=first_write, index=False)
            print(f"💾 Flushed pages {page - CHUNK_SIZE + 1}–{page} to disk ({len(current_chunk)} markets). RAM cleared.")
            current_chunk = []
            first_write = False

        # --- Stop conditions ---
        if not next_cursor or next_cursor == cursor:
            print("Cursor exhausted. Done!")
            break

        cursor = next_cursor
        page += 1
        time.sleep(0.2)

    # --- Flush any remaining markets not yet written ---
    if current_chunk:
        df_chunk = pd.DataFrame(current_chunk)
        df_chunk.to_csv(OUTPUT_FILE, mode='a', header=first_write, index=False)
        print(f"Final flush: {len(current_chunk)} markets written.")

    print(f"\n Complete! Data saved to {OUTPUT_FILE}")

    # --- Return full DataFrame and print shape ---
    markets_df = pd.read_csv(OUTPUT_FILE)
    print(f"Final shape: {markets_df.shape}")
    return markets_df


In [12]:
markets_df = get_all_markets()

  page markets: 100 | chunk: 100 | next_cursor: True
  page markets: 100 | chunk: 200 | next_cursor: True
  page markets: 100 | chunk: 300 | next_cursor: True
  page markets: 100 | chunk: 400 | next_cursor: True
  page markets: 100 | chunk: 500 | next_cursor: True
  page markets: 100 | chunk: 600 | next_cursor: True
  page markets: 100 | chunk: 700 | next_cursor: True
  page markets: 100 | chunk: 800 | next_cursor: True
  page markets: 100 | chunk: 900 | next_cursor: True
  page markets: 100 | chunk: 1000 | next_cursor: True
  page markets: 100 | chunk: 1100 | next_cursor: True
  page markets: 100 | chunk: 1200 | next_cursor: True
  page markets: 100 | chunk: 1300 | next_cursor: True
  page markets: 100 | chunk: 1400 | next_cursor: True
  page markets: 100 | chunk: 1500 | next_cursor: True
  page markets: 100 | chunk: 1600 | next_cursor: True
  page markets: 100 | chunk: 1700 | next_cursor: True
  page markets: 100 | chunk: 1800 | next_cursor: True
  page markets: 100 | chunk: 1900 | n

KeyboardInterrupt: 

In [17]:
# Read the broken CSV in chunks, write to parquet
import pandas as pd

chunks = pd.read_csv(
    "kalshi_historical_markets.csv",
    on_bad_lines='skip',
    engine='python',
    chunksize=10000  # read 10k rows at a time, never loads full file
)

# Write all chunks to a single parquet file
first = True
for chunk in chunks:
    if first:
        chunk.to_parquet("kalshi_markets.parquet", index=False, engine="fastparquet", compression="snappy")
        first = False
    else:
        chunk.to_parquet("kalshi_markets.parquet", index=False, engine="fastparquet", compression="snappy", append=True)

print("Done")

Done


In [21]:
df = pd.read_parquet("kalshi_markets.parquet")
print(df.columns.tolist())
print(df.shape)
print(df.head(2))

['can_close_early', 'close_time', 'created_time', 'custom_strike', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'is_provisional', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'mve_collection_ticker', 'mve_selected_legs', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'status', 'strike_type', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']
(5000, 44)
   can_close_early            close_time                 created_time  \
0             True  2026-06-16T19:07:00Z  2026-06-13T15:14:42.139341Z   


In [22]:
print("\n--- Time Range ---")
print("open_time: ", df['open_time'].min(), "→", df['open_time'].max())
print("close_time:", df['close_time'].min(), "→", df['close_time'].max())

print("\n--- Status Breakdown ---")
print(df['status'].value_counts())

print("\n--- Sample Tickers ---")
print(df['ticker'].head(10).tolist())

print("\n--- Event Ticker Prefixes (proxy for category) ---")
df['series'] = df['event_ticker'].str.split('-').str[0]
print(df['series'].value_counts().head(20))


--- Time Range ---
open_time:  2026-06-13T15:11:29Z → 2026-06-13T15:14:42Z
close_time: 2026-06-13T15:15:00Z → 2026-07-12T04:00:00Z

--- Status Breakdown ---
status
active    4984
closed      16
Name: count, dtype: int64

--- Sample Tickers ---
['KXMVESPORTSMULTIGAMEEXTENDED-S2026956CA2F3A22-6508E9B581E', 'KXMVESPORTSMULTIGAMEEXTENDED-S202690D101AD32B-327FDD87C7E', 'KXMVESPORTSMULTIGAMEEXTENDED-S20260EB395D9151-93A53596A3A', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026F806FC292FB-324EA614BED', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026B01362814BC-3C3BBDE95EE', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026FDA5FB48306-5F9C81F10D9', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026F32A1D4464A-2B4CF0299AA', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026F5861081174-69E7A9350F3', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026CF5A825FB76-ECA629A93DB', 'KXMVESPORTSMULTIGAMEEXTENDED-S2026461A2549D58-CB7C8E9AA11']

--- Event Ticker Prefixes (proxy for category) ---
series
KXMVESPORTSMULTIGAMEEXTENDED    4241
KXMVECROSSCATEGORY               759
Name: count